# Whole-brain simulation with TVB

A resting-state simulation of 76 human regions, built from the generic `NW_TVB_*` nodes.
Each region is a neural mass model; the connectome couples them; two monitors record fast
activity and BOLD.

```
                                   ┌──────────► NW_TVB_Model ── tvb_model_info ──► NW_TVB_Integrator
                                   │                  │                                  │
 NW_TVB_Connectivity ──────────────┤                  ▼                                  ▼
 (data/connectivity_76.zip)        ├────────────► NW_TVB_Simulator ◄────────────── (tvb_integrator)
                                   │               ▲    ▲     ▲
                                   │               │    │     └──── NW_TVB_Coupling
                                   │   NW_TVB_Monitor_fast  NW_TVB_Monitor_bold   (fan-in port tvb_monitors)
                                   │                  │
                                   │                  ├──► NW_TVB_TimeSeriesPlot_fast   (fast activity + spectrum)
                                   │                  ├──► NW_TVB_TimeSeriesPlot_bold   (BOLD signal)
                                   └──────────────────┴──► NW_TVB_FunctionalConnectivity (FC, FCD)
```

This is Example 1 of `TVB_whole_brain_examples.md`, which lists seven more — eight neural mass
models, five integrators and five monitor types — all on **this same graph**, changing parameters
only. The workflow was assembled in the web application; the notebook runs the same ten nodes.

Requires `tvb-library` (`pip install tvb-library`).

## 1. Nodes

In [ ]:
from neuroworkflow import WorkflowBuilder

from neuroworkflow.nodes.network.NW_TVB_Connectivity            import NW_TVB_Connectivity
from neuroworkflow.nodes.network.NW_TVB_Model                   import NW_TVB_Model
from neuroworkflow.nodes.network.NW_TVB_Coupling                import NW_TVB_Coupling
from neuroworkflow.nodes.simulation.NW_TVB_Integrator           import NW_TVB_Integrator
from neuroworkflow.nodes.simulation.NW_TVB_Monitor              import NW_TVB_Monitor
from neuroworkflow.nodes.simulation.NW_TVB_Simulator            import NW_TVB_Simulator
from neuroworkflow.nodes.analysis.NW_TVB_TimeSeriesPlot         import NW_TVB_TimeSeriesPlot
from neuroworkflow.nodes.analysis.NW_TVB_FunctionalConnectivity import NW_TVB_FunctionalConnectivity

In [ ]:
conn      = NW_TVB_Connectivity("NW_TVB_Connectivity")
model     = NW_TVB_Model("NW_TVB_Model")
coupling  = NW_TVB_Coupling("NW_TVB_Coupling")
integrator = NW_TVB_Integrator("NW_TVB_Integrator")
mon_fast  = NW_TVB_Monitor("NW_TVB_Monitor_fast")
mon_bold  = NW_TVB_Monitor("NW_TVB_Monitor_bold")
sim       = NW_TVB_Simulator("NW_TVB_Simulator")
plot_fast = NW_TVB_TimeSeriesPlot("NW_TVB_TimeSeriesPlot_fast")
plot_bold = NW_TVB_TimeSeriesPlot("NW_TVB_TimeSeriesPlot_bold")
fc        = NW_TVB_FunctionalConnectivity("NW_TVB_FunctionalConnectivity")

conn.configure(connectivity_file="./data/connectivity_76.zip")

model.configure(model_type="Generic2dOscillator")
coupling.configure(coupling_type="Scaling", a=0.0075)

# dt=0 and nsig='auto' take the model preset's values, so changing the model
# never leaves an unstable step or a noise vector of the wrong length.
integrator.configure(integrator_type="HeunDeterministic", dt=0.0, nsig="auto")

mon_fast.configure(label="fast")                                    # 1 ms temporal average
mon_bold.configure(label="bold", monitor_type="Bold", period=1000.0)  # TR = 1 s

sim.configure(simulation_length=30000.0, transient=5000.0)

plot_fast.configure(monitor="fast", title="Fast activity", show_spectrum=True,
                    save_to_file=True, output_path="./results/tvb_whole_brain/fast.npz")
plot_bold.configure(monitor="bold", title="BOLD",
                    save_to_file=True, output_path="./results/tvb_whole_brain/bold.npz")
fc.configure(save_to_file=True, output_path="./results/tvb_whole_brain/fc.npz")

## 2. The graph

`tvb_monitors` is a fan-in port: both monitors feed the one simulator, and the simulator returns
its results keyed by monitor label, which is how the two plots and the FC node each pick the
recording they want.

In [ ]:
wf = WorkflowBuilder("NW_TVB_WholeBrain",
                     context={"results_path": "./results/tvb_whole_brain"})

for node in (conn, model, coupling, integrator, mon_fast, mon_bold,
             sim, plot_fast, plot_bold, fc):
    wf.add_node(node)

wf.connect("NW_TVB_Connectivity", "tvb_connectivity", "NW_TVB_Model", "tvb_connectivity")
wf.connect("NW_TVB_Connectivity", "tvb_connectivity", "NW_TVB_Simulator", "tvb_connectivity")
wf.connect("NW_TVB_Connectivity", "tvb_connectivity", "NW_TVB_FunctionalConnectivity", "tvb_connectivity")
wf.connect("NW_TVB_Model", "tvb_model", "NW_TVB_Simulator", "tvb_model")
wf.connect("NW_TVB_Model", "tvb_model_info", "NW_TVB_Integrator", "tvb_model_info")
wf.connect("NW_TVB_Coupling", "tvb_coupling", "NW_TVB_Simulator", "tvb_coupling")
wf.connect("NW_TVB_Integrator", "tvb_integrator", "NW_TVB_Simulator", "tvb_integrator")
wf.connect("NW_TVB_Monitor_fast", "tvb_monitor", "NW_TVB_Simulator", "tvb_monitors")
wf.connect("NW_TVB_Monitor_bold", "tvb_monitor", "NW_TVB_Simulator", "tvb_monitors")
wf.connect("NW_TVB_Simulator", "tvb_results", "NW_TVB_TimeSeriesPlot_fast", "tvb_results")
wf.connect("NW_TVB_Simulator", "tvb_results", "NW_TVB_TimeSeriesPlot_bold", "tvb_results")
wf.connect("NW_TVB_Simulator", "tvb_results", "NW_TVB_FunctionalConnectivity", "tvb_results")

workflow = wf.build()
print(workflow)

## 3. Run

Around 20 seconds. The nodes draw as they go: the connectome matrices, the fast traces with
their power spectrum, the BOLD traces, and the FC/FCD matrices.

In [ ]:
assert workflow.execute(), "workflow failed — check the printed errors above"
print("\ndone")

## 4. What came out

In [ ]:
metrics = fc._output_ports["fc_metrics"].value
for key, value in metrics.items():
    print(f"  {key:20} {value}")

### Reading the result

With `a = 0.0075` the network partly synchronises the oscillators, so FC is high (mean ≈ 0.71)
and only loosely shaped by the connectome (FC–SC ≈ 0.21).

Three things to try, each one parameter:

```python
integrator.configure(integrator_type="HeunStochastic")   # preset noise: less regular, lower FC
coupling.configure(a=0.0)                                # regions decouple, FC near 0
coupling.configure(a=0.05)                               # stronger synchrony
```

Then `workflow.execute()` again — no rebuild. Give each run its own `output_path` if you want to
keep the `.npz` files side by side.

## 5. The other seven examples

`TVB_whole_brain_examples.md` holds a table of every parameter that changes between examples, and
what to expect from each: MontbrioPazoRoxin bistable up/down states, ReducedWongWangExcInh,
JansenRit with conduction delays, SupHopf, WilsonCowan, and two Epileptor variants with a seizure
onset zone. Nodes and connections never change — only the values set in section 1.